# 03 - Explore the raw data
This notebook contains read-only exploration. It is not needed to reproduce the data work.
Run after 02_load_tennis_bigquery

## Install the pinned libraries

In [ ]:

REQUIREMENTS_URL = "https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Helpers

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.max_columns", 100, "display.width", 200)
TABLE = f"{PROJECT_ID}.tennis_raw.wta_matches"

## Row overview

In [ ]:
run(f"SELECT * FROM `{TABLE}` LIMIT 3").T

## Row count and duplicates
43,924 rows, no duplicates

In [ ]:
run(f"""
SELECT COUNT(*) AS rows_total,
       COUNT(DISTINCT CONCAT(tourney_id, '|', match_num)) AS unique_matches
FROM `{TABLE}`
""")

## Missing values
**Findings:**
- Serve statistics (aces, double faults etc) are approx11% empty overall.
- `minutes` is approx 41% empty.
- Seed and entry columns are usually empty because most players are unseeded or entered directly, so empty are not true missing values
- Height is 10.5% (winner) and 14.9% (loser) empty.
- Rank and ranking points are 1.4% to 2.8% empty (unranked players?).
- `score` is fully filled in, so games and sets can be derived from it.

In [ ]:
cols = [f.name for f in bq.get_table(TABLE).schema]
exprs = ",\n  ".join(
    f"ROUND(100 * COUNTIF(`{c}` IS NULL OR `{c}` = '') / COUNT(*), 1) AS `{c}`" for c in cols
)
missing = run(f"SELECT {exprs} FROM `{TABLE}`").T
missing.columns = ["percent_empty"]
missing.sort_values("percent_empty", ascending=False)

## Is there a real match date?
No - every tournament has exactly one `tourney_date`, the start of the tournament. Match days have to be estimated later.

In [ ]:
run(f"""
SELECT COUNT(DISTINCT tourney_id) AS tournaments,
       COUNTIF(n_dates > 1) AS tournaments_with_more_than_one_date
FROM (
  SELECT tourney_id, COUNT(DISTINCT tourney_date) AS n_dates
  FROM `{TABLE}` GROUP BY tourney_id
)
""")

## Tournament levels, surfaces and names

- 2,529 tournament IDs but 1,633 distinct names.
- Level D is the team competition (Fed Cup / Billie Jean King Cup). Team format at national venues, so it is excluded.
- Surface has inconsistent case (`Clay` vs `clay`)
- 115 team-event rows have no surface.

In [ ]:
print(run(f"""
SELECT tourney_level, surface, COUNT(*) AS matches
FROM `{TABLE}` GROUP BY 1, 2 ORDER BY matches DESC
"""))

print(run(f"""
SELECT COUNT(DISTINCT tourney_name) AS distinct_names,
       COUNT(DISTINCT tourney_id) AS distinct_tourney_ids
FROM `{TABLE}`
"""))

## Investigating the small levels
- `O` = Olympics; `W` = WTA 125-type events (plus one mislabelled 2015 "WTA Finals"); `35+H` and `50+H` = lower-tier ITF events.
- All are small, mostly one-off editions, so they are excluded.

In [ ]:
run(f"""
SELECT tourney_level, tourney_name, COUNT(*) AS matches,
       MIN(tourney_date) AS first_date, MAX(tourney_date) AS last_date
FROM `{TABLE}`
WHERE tourney_level IN ('W','O','50+H','35+H')
GROUP BY 1, 2 ORDER BY tourney_level, matches DESC
""")

## Examine missing values by level and by year
- `minutes` is a 2016-onward feature only.
- Serve statistics are 10-13% empty in 2010 to 2014, 22% in 2015, and 0-2% from 2016.
- 2026 is about 13% empty (partial year, 2016 not over yet)
- Level D has 62% of serve statistics missing.
- The gaps follow time - need to take care with split

In [ ]:
print(run(f"""
SELECT tourney_level, COUNT(*) AS matches,
       ROUND(100*COUNTIF(w_svpt IS NULL)/COUNT(*),1) AS pct_no_serve_stats,
       ROUND(100*COUNTIF(minutes IS NULL)/COUNT(*),1) AS pct_no_minutes
FROM `{TABLE}` GROUP BY tourney_level ORDER BY matches DESC
"""))

print(run(f"""
SELECT SUBSTR(tourney_date,1,4) AS year, COUNT(*) AS matches,
       ROUND(100*COUNTIF(w_svpt IS NULL)/COUNT(*),1) AS pct_no_serve_stats,
       ROUND(100*COUNTIF(minutes IS NULL)/COUNT(*),1) AS pct_no_minutes
FROM `{TABLE}` WHERE tourney_level IN ('G','PM','P','I')
GROUP BY year ORDER BY year
"""))

## Can the tournament ID identify a venue?
No -  some IDs cover different cities  and some tournaments change ID. Names are more stable.

In [ ]:
print(run(f"""
SELECT tourney_level,
       COUNT(DISTINCT tourney_id) AS tourney_ids,
       COUNT(DISTINCT SUBSTR(tourney_id, 6)) AS distinct_id_suffixes,
       COUNT(DISTINCT tourney_name) AS distinct_names
FROM `{TABLE}`
WHERE tourney_level IN ('G','PM','P','I')
GROUP BY tourney_level ORDER BY tourney_ids DESC
"""))

run(f"""
SELECT SUBSTR(tourney_id, 6) AS id_suffix,
       COUNT(DISTINCT SUBSTR(tourney_id, 1, 4)) AS years,
       ARRAY_AGG(DISTINCT tourney_name LIMIT 4) AS example_names
FROM `{TABLE}` WHERE tourney_level IN ('G','PM','P','I')
GROUP BY id_suffix ORDER BY years DESC LIMIT 15
""")

## What kinds of score are there?
 37,542 normal scores, 1,173 retirements (`RET`), 261 walkovers (`W/O`) and 22 defaults (`DEF`).

In [ ]:
run(f"""
SELECT COALESCE(REGEXP_EXTRACT(score, r'([A-Za-z]+(?:/[A-Za-z]+)?)'), '(normal score)') AS tag,
       COUNT(*) AS matches
FROM `{TABLE}`
WHERE tourney_level IN ('G','PM','P','I')
GROUP BY tag ORDER BY matches DESC
""")

## Which events have a round-robin stage?
 Rule used in notebook 04: drop every event that has a round-robin stage.

In [ ]:
run(f"""
SELECT LOWER(TRIM(tourney_name)) AS name_key, tourney_level, COUNT(DISTINCT tourney_id) AS editions, COUNT(*) AS matches
FROM `{TABLE}`
WHERE round = 'RR' AND tourney_level IN ('G','PM','P','I')
GROUP BY 1, 2 ORDER BY 1
""")

## Rounds and draw sizes
This is what notebook 05 uses to estimate match days.

In [ ]:
print(run(f"""
SELECT tourney_level, round, COUNT(*) AS matches, COUNT(DISTINCT tourney_id) AS tournaments
FROM `{TABLE}` WHERE tourney_level IN ('G','PM','P','I')
GROUP BY 1, 2 ORDER BY 1, matches DESC
"""))

print(run(f"""
SELECT tourney_level, draw_size, COUNT(DISTINCT tourney_id) AS tournaments
FROM `{TABLE}` WHERE tourney_level IN ('G','PM','P','I')
GROUP BY 1, 2 ORDER BY 1, 2
"""))

## Values in the seed columns that are not numbers
 97 values (winner: Q 15, WC 10; loser: Q 44, WC 20, LL 8) are entry codes (qualifier, wild card, lucky loser) sitting in the seed column. None of those rows had an entry value already. Notebook 04 moves them to the entry column.

In [ ]:
run(f"""
SELECT 'winner' AS side, TRIM(winner_seed) AS code, NULLIF(TRIM(winner_entry), '') AS existing_entry, COUNT(*) AS n
FROM `{TABLE}`
WHERE tourney_level IN ('G','PM','P','I')
  AND NULLIF(TRIM(winner_seed), '') IS NOT NULL AND SAFE_CAST(TRIM(winner_seed) AS INT64) IS NULL
GROUP BY 2, 3
UNION ALL
SELECT 'loser', TRIM(loser_seed), NULLIF(TRIM(loser_entry), ''), COUNT(*)
FROM `{TABLE}`
WHERE tourney_level IN ('G','PM','P','I')
  AND NULLIF(TRIM(loser_seed), '') IS NOT NULL AND SAFE_CAST(TRIM(loser_seed) AS INT64) IS NULL
GROUP BY 2, 3
ORDER BY n DESC
""")

## Tournaments whose start date falls in a different year from their season
The year in `tourney_id` is the season the source assigns. A tournament that starts in the last days of December can belong to the next season, so its `tourney_date` year differs from its `tourney_id` year

In [ ]:
run(f"""
SELECT SUBSTR(tourney_id, 1, 4) AS id_year, SUBSTR(tourney_date, 1, 4) AS date_year,
       tourney_name, tourney_date, COUNT(*) AS matches
FROM `{TABLE}`
WHERE SUBSTR(tourney_id, 1, 4) <> SUBSTR(tourney_date, 1, 4)
GROUP BY 1, 2, 3, 4 ORDER BY tourney_date
""")